# Agent Memory: a Coding Project Assistant

This notebook demonstrates why ordinary chat history is not durable memory, then adds an optional `memory.md` file. The assistant is deliberately small: every request receives a normal answer, then a second LLM call examines only the user's message to decide whether a durable preference or project fact should be saved.

**Model:** `deepseek/deepseek-v4.1-flash` through OpenRouter.

In [1]:
import json
import os
import re
import difflib
import html
from pathlib import Path

import markdown
import requests
from dotenv import load_dotenv, find_dotenv
from IPython.display import Markdown, HTML, display

is_env_var_loaded = load_dotenv(find_dotenv())
print(f"Is environment variables loaded?: {'Yes' if is_env_var_loaded else 'No'}")

MODEL = 'deepseek/deepseek-v4.1-flash'
OPEN_ROUTER_KEY = os.environ.get('OPEN_ROUTER_KEY')
CHAT_URL = 'https://openrouter.ai/api/v1/chat/completions'
headers = {
    'Authorization': f'Bearer {OPEN_ROUTER_KEY}',
    'Content-Type': 'application/json',
}

Is environment variables loaded?: Yes


## Helper Functions

In [2]:
def call_llm(messages, temperature=0.2):
    payload = {'model': MODEL, 'messages': messages, 'temperature': temperature}
    response = requests.post(CHAT_URL, headers=headers, json=payload, timeout=90)
    response.raise_for_status()
    return response.json()['choices'][0]['message']['content']

In [3]:
MEMORY_FILE = Path('memory.md')
EMPTY_MEMORY = '''# Coding Project Assistant Memory

## Personal
- No saved details yet.

## General Preferences
- No saved preferences yet.

## Technology Preferences
- No saved technology preferences yet.

## Projects
- No saved projects yet.
'''

def reset_memory():
    MEMORY_FILE.write_text(EMPTY_MEMORY, encoding='utf-8')

def read_memory():
    if not MEMORY_FILE.exists():
        reset_memory()
    return MEMORY_FILE.read_text(encoding='utf-8')

reset_memory()

In [4]:
def memory_decision(user_message, old_memory):
    prompt = f'''You maintain a Markdown memory file for a Coding Project Assistant.
Update it only with durable, useful facts explicitly stated by the user: identity details, answer-style preferences, technology preferences, and active coding-project facts. 

Do not save transient requests, assistant suggestions, secrets, or guesses.

Keep memory extremely compact: use keyword-like bullets of at most 10 words, merge related facts into one bullet, and rewrite existing verbose bullets when updating. Aim for no more than two meaningful bullets per section. Prefer `Answer style: code only; simple examples.` over a sentence, `Project: CSV analytics → PostgreSQL.` over implementation detail, and `Python tooling: uv; pre-commit.` over separate bullets. If a new preference conflicts with an old one in the same category, replace the old preference; never retain both. For example, replace `Database: PostgreSQL.` with `Database: Oracle.` Preserve the four exact headings and bullet-list style.

Current memory:
---
{old_memory}
---

User message: {user_message}
Return JSON only, with keys `should_update` (boolean), `reason` (short string), and `memory` (the complete Markdown file; unchanged if should_update is false).'''
    
    raw = call_llm([{'role': 'system', 'content': 'Return valid JSON only.'}, {'role': 'user', 'content': prompt}])
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw.strip())
    try:
        decision = json.loads(cleaned)
        assert isinstance(decision['should_update'], bool)
        assert isinstance(decision['memory'], str)
        return decision
    except (json.JSONDecodeError, KeyError, AssertionError) as error:
        raise ValueError(f'Memory-decision response was not valid JSON: {raw}') from error

## The Coding Project Assistant

In [5]:
ASSISTANT_SYSTEM = '''You are a concise Coding Project Assistant. Help the user build data and software projects. Do not make assumptions.'''

class PersonalProjectAssistant:
    def __init__(self, memory_enabled=False):
        self.memory_enabled = memory_enabled
        self.history = []

    def ask(self, user_message):
        messages = [{'role': 'system', 'content': ASSISTANT_SYSTEM}]
        if self.memory_enabled:
            messages.append({'role': 'system', 'content': 'Saved memory (User Preferences):\n' + read_memory()})
        messages.extend(self.history) ## Previous turns
        messages.append({'role': 'user', 'content': user_message})
        reply = call_llm(messages)
        self.history += [{'role': 'user', 'content': user_message}, {'role': 'assistant', 'content': reply}]
        decision = {'should_update': False, 'reason': 'Memory is disabled.', 'memory': read_memory()}
        if self.memory_enabled:
            old_memory = read_memory()
            # Memory is derived only from what the user explicitly said.
            decision = memory_decision(user_message, old_memory)
            if decision['should_update']:
                MEMORY_FILE.write_text(decision['memory'].strip() + '\n', encoding='utf-8')
        # Render the LLM response by itself so fenced code blocks display correctly.
        display(Markdown(f'### User\n\n{user_message}\n\n### Assistant\n\n{reply}'))
        #print('Memory decision:', decision['reason'])

## 1. Without memory

The first conversation establishes a preference. The second is a brand-new assistant instance, so it has no history and—because memory is off—no file context either.

In [7]:
stateless_chat_1 = PersonalProjectAssistant()
stateless_chat_1.ask('Write Python code that combines all CSV files in a data folder and groups sales by month.')

### User

Write Python code that combines all CSV files in a data folder and groups sales by month.

### Assistant

Before I give you code, I need a few details to avoid guessing:

1. **Column names** — what is the date column called, and what is the sales column called? (e.g., `order_date`, `amount`)
2. **Date format** — is it `2024-01-15`, `01/15/2024`, or something else?
3. **CSV structure** — do all files have the same columns and a header row?
4. **Folder path** — where is the data folder relative to the script?
5. **Output** — do you want the monthly totals printed, saved to a CSV, or both?

Here's a template with the assumptions clearly marked so you can adjust:

```python
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")          # <-- adjust folder path
DATE_COL = "order_date"          # <-- adjust to your date column
SALES_COL = "amount"             # <-- adjust to your sales column

# Combine all CSVs in the folder
files = sorted(DATA_DIR.glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR.resolve()}")

df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

# Parse dates and group by month
df[DATE_COL] = pd.to_datetime(df[DATE_COL])   # add format=... if needed
monthly = (
    df.groupby(df[DATE_COL].dt.to_period("M"))[SALES_COL]
      .sum()
      .reset_index()
      .rename(columns={DATE_COL: "month"})
)

print(monthly)
monthly.to_csv("monthly_sales.csv", index=False)
```

If you tell me the actual column names and date format, I'll tailor it exactly — including handling mixed date formats or files with inconsistent columns if that's a concern.

In [8]:
stateless_chat_1.ask('This is a large collection of CSVs, so I prefer Dask DataFrames over pandas. Please rewrite the code.')

### User

This is a large collection of CSVs, so I prefer Dask DataFrames over pandas. Please rewrite the code.

### Assistant

```python
import dask.dataframe as dd
from pathlib import Path

DATA_DIR = Path("data")          # <-- adjust folder path
DATE_COL = "order_date"          # <-- adjust to your date column
SALES_COL = "amount"             # <-- adjust to your sales column

files = sorted(DATA_DIR.glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR.resolve()}")

# Dask reads all files lazily; parse dates up front
df = dd.read_csv(
    [str(f) for f in files],
    parse_dates=[DATE_COL],      # add date_format=... if your format is non-standard
    assume_missing=True,         # helps when dtypes differ across files
)

# Group by year + month (Dask doesn't support dt.to_period)
monthly = (
    df.groupby([df[DATE_COL].dt.year, df[DATE_COL].dt.month])[SALES_COL]
      .sum()
      .reset_index()
      .rename(columns={DATE_COL: "month"})
)

# Result is tiny — bring it to pandas
monthly_pd = monthly.compute().sort_values(["year", "month"])
print(monthly_pd)
monthly_pd.to_csv("monthly_sales.csv", index=False)
```

Notes on the Dask-specific changes:

- **`dt.to_period("M")` isn't supported** in Dask, so I group by `year` + `month` instead. If you want a single `YYYY-MM` label, add `monthly_pd["month"] = pd.to_datetime(monthly_pd[["year","month"]].assign(day=1)).dt.strftime("%Y-%m")` after `.compute()`.
- **`assume_missing=True`** avoids errors when some files have all-null columns (common in large CSV collections).
- **`parse_dates` at read time** is cheaper than converting later, since Dask pushes it into each partition.
- If the collection is huge, consider `df = df.persist()` before the groupby, or `dd.read_csv(..., blocksize="256MiB")` to tune partition size.

Still need from you to finalize: actual column names, date format, and whether files share identical schemas (if not, `assume_missing` may not be enough and we'd need `dtype=` overrides).

In [18]:
# A new conversation: history is empty and memory is disabled.
stateless_chat_2 = PersonalProjectAssistant()
stateless_chat_2.ask('Write code to load all monthly CSV files, combine them, and calculate total sales by product.')

### User

Write code to load all monthly CSV files, combine them, and calculate total sales by product.

### Assistant

Here's a pandas solution. I've marked the parts you'll likely need to adjust — see the questions at the bottom.

```python
import glob
import os
import pandas as pd

DATA_DIR = "data/monthly"      # folder holding the monthly CSVs
PATTERN  = "*.csv"             # e.g. "sales_2024_*.csv" if files are named by month

PRODUCT_COL = "product"        # <-- your product column name
SALES_COL   = "sales"          # <-- your sales/amount column name

# 1. Find the files
files = sorted(glob.glob(os.path.join(DATA_DIR, PATTERN)))
if not files:
    raise FileNotFoundError(f"No files matched {PATTERN} in {DATA_DIR}")

# 2. Load each one, tagging where it came from
frames = []
for path in files:
    df = pd.read_csv(path)
    df["source_file"] = os.path.basename(path)
    frames.append(df)

# 3. Combine
combined = pd.concat(frames, ignore_index=True)

# Normalize headers so "Product Name" and "product_name" line up
combined.columns = [c.strip().lower().replace(" ", "_") for c in combined.columns]

# 4. Clean the sales column (bad values become NaN instead of crashing)
combined[SALES_COL] = pd.to_numeric(combined[SALES_COL], errors="coerce")

# 5. Total by product
totals = (
    combined.groupby(PRODUCT_COL, as_index=False)[SALES_COL]
    .sum()
    .sort_values(SALES_COL, ascending=False)
)

totals.to_csv("total_sales_by_product.csv", index=False)
print(totals)
```

**Things I assumed — please confirm or correct:**

1. **File location/naming** — all monthly CSVs sit in one folder and share a pattern. If they're spread across subfolders, use `glob.glob(..., recursive=True)` with `**/*.csv`.
2. **Column names** — I guessed `product` and `sales`. What are they actually?
3. **Same schema across files** — I assume every monthly file has identical columns. If not, `pd.concat` will produce NaNs for missing ones.
4. **"Total sales" means** — sum of a revenue/amount column. If instead you have `quantity` × `unit_price`, or need to net out returns/refunds, that changes the math.
5. **Product identity** — I'm grouping on the raw product string. If the same product appears with different casing, spacing, or SKUs across months, you'll get split rows and need a normalization or SKU mapping step.
6. **Duplicates** — if a month's file can be loaded twice (e.g. a corrected re-export), you'll double-count. Worth a dedupe key if that's possible.

Tell me the real column names and file layout and I'll tighten this up.

## 2. With memory

In [19]:
memory_chat_1 = PersonalProjectAssistant(memory_enabled=True)
memory_chat_1.ask('Write Python code with pandas that combines all CSV files in a data folder and groups sales by month.')
memory_chat_1.ask('This is a large collection of CSVs, so I prefer Dask DataFrames over pandas. Please rewrite the code.')

### User

Write Python code with pandas that combines all CSV files in a data folder and groups sales by month.

### Assistant

```python
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
SALES_COL = "sales"   # <-- set to your sales column name
DATE_COL = "date"     # <-- set to your date column name

files = sorted(DATA_DIR.glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR.resolve()}")

frames = []
for f in files:
    df = pd.read_csv(f)
    df["__source_file"] = f.name   # keeps provenance; drop later if unwanted
    frames.append(df)

combined = pd.concat(frames, ignore_index=True)

# Parse dates; report and drop rows that can't be parsed
combined[DATE_COL] = pd.to_datetime(combined[DATE_COL], errors="coerce")
bad = combined[DATE_COL].isna().sum()
if bad:
    print(f"Warning: dropping {bad} rows with unparseable dates")
combined = combined.dropna(subset=[DATE_COL])

monthly = (
    combined.groupby(combined[DATE_COL].dt.to_period("M"))[SALES_COL]
    .sum()
    .reset_index()
    .rename(columns={DATE_COL: "month"})
)
monthly["month"] = monthly["month"].dt.to_timestamp()

print(monthly)
monthly.to_csv("monthly_sales.csv", index=False)
```

Notes / things to check before running:

- **Column names**: `SALES_COL` and `DATE_COL` are placeholders — set them to match your files. If column names differ across files, `pd.concat` will produce NaN-filled columns; normalize names in the loop first.
- **Sales dtype**: if the sales column contains currency symbols or thousands separators (e.g. `"1,234.50"`), clean it before grouping: `combined[SALES_COL] = pd.to_numeric(combined[SALES_COL].astype(str).str.replace(r"[,$]", "", regex=True), errors="coerce")`.
- **Duplicates**: if files overlap in date ranges, rows will be double-counted. Consider `combined.drop_duplicates()` or a dedup key.
- **Large data**: you've mentioned preferring Dask for large CSV collections. If these files are big, the same logic maps directly to `dask.dataframe` (`dd.read_csv("data/*.csv")`, `groupby(df.date.dt.to_period("M"))["sales"].sum().compute()`).

### User

This is a large collection of CSVs, so I prefer Dask DataFrames over pandas. Please rewrite the code.

### Assistant

```python
from pathlib import Path
import dask.dataframe as dd

DATA_DIR = Path("data")
SALES_COL = "sales"   # <-- set to your sales column name
DATE_COL = "date"     # <-- set to your date column name

files = sorted(str(p) for p in DATA_DIR.glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR.resolve()}")

df = dd.read_csv(files, assume_missing=True)

# Clean sales if it may contain currency symbols / thousands separators
df[SALES_COL] = dd.to_numeric(
    df[SALES_COL].astype(str).str.replace(r"[,$]", "", regex=True),
    errors="coerce",
)

# Parse dates; drop unparseable rows
df[DATE_COL] = dd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=[DATE_COL, SALES_COL])

# Dask doesn't support dt.to_period in groupby, so derive year/month columns
df = df.assign(
    year=df[DATE_COL].dt.year,
    month=df[DATE_COL].dt.month,
)

monthly = (
    df.groupby(["year", "month"])[SALES_COL]
    .sum()
    .reset_index()
    .compute()
    .sort_values(["year", "month"])
    .reset_index(drop=True)
)

print(monthly)
monthly.to_csv("monthly_sales.csv", index=False)
```

Notes:

- **`assume_missing=True`** lets Dask treat columns that are sometimes empty as nullable — useful when CSVs have inconsistent schemas. If schemas are truly identical, you can drop it for speed.
- **`dt.to_period` isn't supported** in Dask groupby, hence the `year`/`month` columns. If you want a single timestamp column, add `monthly["month_start"] = pd.to_datetime(dict(year=monthly.year, month=monthly.month, day=1))` after `.compute()`.
- **Column names**: `SALES_COL` and `DATE_COL` are placeholders — set them to match your files. If names differ across files, normalize them before `read_csv` (e.g. read each file's header first, or use `columns=` with a rename map).
- **Duplicates**: if files overlap in date ranges, rows will be double-counted. Add `df = df.drop_duplicates()` before grouping if that's a risk.
- **Tuning**: for a large collection, consider `dd.read_csv(files, blocksize="256MiB")` (or larger) and a `LocalCluster`/`Client` with more workers to control partition count.

In [20]:
# Another genuinely new conversation; only memory connects it to the earlier one.
memory_chat_2 = PersonalProjectAssistant(memory_enabled=True)
memory_chat_2.ask('Write Python code to load all monthly CSV files and calculate total sales by product.')

### User

Write Python code to load all monthly CSV files and calculate total sales by product.

### Assistant

Here's a Dask-based version (per your preference for Dask over pandas on large CSV collections). I've marked the values I had to guess — swap them for your actual names.

```python
from pathlib import Path
import dask.dataframe as dd

# ---- CONFIG: adjust these to match your data ----
DATA_DIR    = Path("data/monthly")      # folder holding the monthly CSVs
FILE_GLOB   = "sales_*.csv"             # e.g. sales_2024-01.csv, sales_2024-02.csv
PRODUCT_COL = "product_id"              # column identifying the product
SALES_COL   = "sales_amount"            # column holding the sales value
OUT_PATH    = Path("total_sales_by_product.csv")
# ------------------------------------------------

paths = sorted(DATA_DIR.glob(FILE_GLOB))
if not paths:
    raise FileNotFoundError(f"No files matched {DATA_DIR / FILE_GLOB}")

print(f"Loading {len(paths)} files...")

ddf = dd.read_csv(
    [str(p) for p in paths],
    dtype={PRODUCT_COL: "string", SALES_COL: "float64"},
    assume_missing=True,   # treat empty strings as NaN in object columns
)

totals = (
    ddf.groupby(PRODUCT_COL)[SALES_COL]
       .sum()
       .to_frame("total_sales")
       .reset_index()
       .compute()                       # single pass over all files
       .sort_values("total_sales", ascending=False)
)

totals.to_csv(OUT_PATH, index=False)
print(f"Wrote {len(totals):,} products to {OUT_PATH}")
print(totals.head(20))
```

Notes:
- `dd.read_csv` accepts a list of paths or a glob string, so it reads all months in one lazy graph and only materializes the grouped result.
- If your files are large, add `blocksize="256MiB"` (or similar) to `read_csv` to control partition size.
- If a monthly file has a header row repeated, Dask handles that fine; if any file lacks a header, pass `header=None` and supply `names=[...]`.

To make this exact rather than a template, I need to know:
1. The actual directory and filename pattern of the monthly files.
2. The exact column names for product and sales.
3. Whether "sales" is a single amount column, or needs to be derived (e.g. `quantity * unit_price`).
4. Whether product IDs are consistent across files (same key format), or need normalization.

## 3. How are we maintaining memory for our agent?

The agent needs a small, durable, human-readable store that is available to every new conversation. In this demo, we introduce a `memory.md` file. The memory-decision call can add or refine bullets beneath four stable sections: **Personal**, **General Preferences**, **Technology Preferences**, and **Projects**.

In [22]:
print(read_memory())

---
---
# Coding Project Assistant Memory

## Personal
- No saved details yet.

## General Preferences
- No saved preferences yet.

## Technology Preferences
- Python; Dask DataFrames over pandas.

## Projects
- Large CSV collection; Dask.

---

---



## 4. See each memory update

A full-file view and a color-highlighted unified diff make each persistence decision visible after every interaction. The earlier chats stored Dask under **Technology Preferences**. The fresh conversations below add personal details, answer-style preferences, extra tooling choices, and project facts under the other sections.

In [23]:
def show_full_memory_diff(before, after):
    """Show the complete current file, then a readable single-column diff."""
    display(Markdown('#### Complete `memory.md` after this update'))
    #display(Markdown(f'```markdown\n{after}\n```'))

    diff_lines = difflib.unified_diff(
        before.splitlines(), after.splitlines(),
        fromfile='memory.md (before)', tofile='memory.md (after)',
        lineterm='', n=10_000,
    )
    rendered_lines = []
    for line in diff_lines:
        safe_line = html.escape(line) or '&nbsp;'
        if line.startswith('+') and not line.startswith('+++'):
            css_class = 'memory-diff-added'
        elif line.startswith('-') and not line.startswith('---'):
            css_class = 'memory-diff-removed'
        elif line.startswith('@@'):
            css_class = 'memory-diff-hunk'
        else:
            css_class = 'memory-diff-neutral'
        rendered_lines.append(f'<span class="{css_class}">{safe_line}</span>')

    styles = '''
    <style>
      .memory-diff { border: 1px solid #d0d7de; border-radius: 6px; padding: 12px; background: #f6f8fa; overflow-x: auto; }
      .memory-diff span { display: block; font-family: var(--jp-code-font-family, monospace); white-space: pre; }
      .memory-diff-added { background: #dafbe1; color: #116329; }
      .memory-diff-removed { background: #ffebe9; color: #cf222e; }
      .memory-diff-hunk { color: #0969da; }
      .memory-diff-neutral { color: #57606a; }
    </style>
    '''
    display(Markdown('#### What changed'))
    display(HTML(styles + '<div class="memory-diff">' + ''.join(rendered_lines) + '</div>'))

def ask_and_show_memory_diff(agent, message):
    before = read_memory()
    agent.ask(message)
    after = read_memory()
    if before == after:
        display(Markdown('*No memory-file change.*'))
    else:
        show_full_memory_diff(before, after)
    return None

In [24]:
reset_memory()

### New Personal Detail

In [25]:
# New conversation 1: Personal
memory_chat_3 = PersonalProjectAssistant(memory_enabled=True)
ask_and_show_memory_diff(memory_chat_3, 'I am Sunny, and I am learning data engineering.')

### User

I am Sunny, and I am learning data engineering.

### Assistant

Nice to meet you, Sunny. I'll keep in mind that you're learning data engineering.

A few things that would help me help you well:

1. **What are you working on right now?** A specific project, a course exercise, or just exploring?
2. **What tools are you using or want to use?** (e.g., Python, SQL, Airflow, dbt, Spark, cloud platform)
3. **What's your current comfort level?** Beginner across the board, or do you already have some Python/SQL background?

Tell me where you're at and what you'd like to build, and we can go from there.

#### Complete `memory.md` after this update

#### What changed

### New Technology Preference

In [26]:
# New conversation 4: Technology Preferences
memory_chat_6 = PersonalProjectAssistant(memory_enabled=True)
ask_and_show_memory_diff(memory_chat_6, 'For Python projects, I prefer uv for package management and pycodestyle for formatting checks. For relational databases, I prefer PostgreSQL.')

### User

For Python projects, I prefer uv for package management and pycodestyle for formatting checks. For relational databases, I prefer PostgreSQL.

### Assistant

Got it — noted for future work:

- **Python projects:** `uv` for package management, `pycodestyle` for formatting checks
- **Relational databases:** PostgreSQL

I'll apply these by default unless you say otherwise. Anything you'd like to start on?

#### Complete `memory.md` after this update

#### What changed

### NO Preference CASE

In [27]:
# New conversation 8: A transient question — no memory update
memory_chat_10 = PersonalProjectAssistant(memory_enabled=True)
ask_and_show_memory_diff(memory_chat_10, 'How do I list files in the current directory with Python?')

### User

How do I list files in the current directory with Python?

### Assistant

Use `pathlib` (modern, recommended) or `os`:

```python
from pathlib import Path

# Names only
for entry in Path(".").iterdir():
    print(entry.name)

# Files only (skip directories)
for entry in Path(".").iterdir():
    if entry.is_file():
        print(entry.name)

# Recursive glob for a pattern
for entry in Path(".").rglob("*.py"):
    print(entry)
```

With `os`:

```python
import os

for name in os.listdir("."):
    print(name)
```

Notes:
- `Path(".").iterdir()` and `os.listdir(".")` are non-recursive; use `rglob`/`os.walk` for recursion.
- `iterdir()` yields `Path` objects, so you get `.name`, `.is_file()`, `.stat()` etc. for free.
- Hidden files (dotfiles) are included by default in both.

*No memory-file change.*

### Update Existing Preference CASE

In [29]:
# New conversation 7: Changed Technology Preference
memory_chat_9 = PersonalProjectAssistant(memory_enabled=True)
ask_and_show_memory_diff(memory_chat_9, 'I will be using Oracle Database instead of PostgreSQL going forward.')

### User

I will be using Oracle Database instead of PostgreSQL going forward.

### Assistant

Got it — I'll note that you're switching to Oracle Database going forward instead of PostgreSQL.

A couple of quick clarifications so I can help you correctly:

1. **Oracle version** — e.g., 19c, 21c, 23ai? (Affects available features like JSON, `FETCH FIRST`, etc.)
2. **Python driver** — `oracledb` (the modern one, formerly cx_Oracle) is the standard choice. Want me to default to that?
3. **Connection style** — thin mode (pure Python, no Oracle client needed) or thick mode (requires Oracle Client libraries)?

Once you confirm, I'll use Oracle conventions in any SQL, schema, or connection code I write for you (e.g., `VARCHAR2`, `NUMBER`, `DATE`/`TIMESTAMP`, sequences/identity columns, `ROWNUM`/`FETCH FIRST`, `DUAL`, etc.).

#### Complete `memory.md` after this update

#### What changed

### Project Preference Update CASE

In [30]:
# New conversation 6: Projects
memory_chat_8 = PersonalProjectAssistant(memory_enabled=True)
ask_and_show_memory_diff(memory_chat_8, 'The CSV analytics pipeline will run nightly and publish its metrics to a dashboard for the sales team.')

### User

The CSV analytics pipeline will run nightly and publish its metrics to a dashboard for the sales team.

### Assistant

Got it — nightly CSV → metrics → sales dashboard. Before I sketch anything, a few things I'd need to pin down:

**Inputs**
- Where do the CSVs land (SFTP, S3, shared drive, email)? One file per night, or a batch?
- Is the schema stable, or does it drift? Any known encoding/quoting quirks?
- Rough volume — rows and file size per night?

**Processing**
- Which metrics matter to sales (e.g. revenue by rep/region, order counts, conversion)? Are the definitions already agreed with the sales team?
- Does the pipeline need to be idempotent — i.e. can a night be re-run safely?
- Should raw CSVs be archived, or is the transformed output enough?

**Storage & serving**
- You're on Oracle — is that the landing/warehouse layer here, or is something else in play?
- How does the dashboard read the metrics: direct Oracle connection, a view, an API, or a file drop?

**Operations**
- What schedules it — cron, Airflow, Oracle DBMS_SCHEDULER, something else?
- What happens on failure? Who gets alerted, and does the dashboard show stale data or nothing?
- Any data-quality gates before publish (row counts, null checks, reconciliation)?

**Stack**
- I'd default to Python managed with `uv` and `pycodestyle` for linting, per your usual setup — say if you'd rather not.

Also: want me to save this as a project in memory? If so, what should I call it, and is the dashboard tool already chosen?

#### Complete `memory.md` after this update

#### What changed

### General Coding Preferences

In [31]:
# New conversation 2: General Preferences
memory_chat_4 = PersonalProjectAssistant(memory_enabled=True)
ask_and_show_memory_diff(memory_chat_4, 'When I ask for code, I prefer code only with no documentation, and use simple examples.')

### User

When I ask for code, I prefer code only with no documentation, and use simple examples.

### Assistant

Noted. I'll keep code responses to code only, no documentation, and use simple examples.

#### Complete `memory.md` after this update

#### What changed

In [32]:
# New conversation 5: General Preferences
memory_chat_7 = PersonalProjectAssistant(memory_enabled=True)
ask_and_show_memory_diff(memory_chat_7, 'Please include type hints in Python examples and use pytest when you show tests.')

### User

Please include type hints in Python examples and use pytest when you show tests.

### Assistant

Noted — type hints on all Python examples, pytest for tests.

```python
# metrics.py
from collections.abc import Iterable


def nightly_total(rows: Iterable[dict[str, float]]) -> float:
    return sum(row["amount"] for row in rows)
```

```python
# test_metrics.py
import pytest

from metrics import nightly_total


def test_nightly_total_sums_amounts() -> None:
    rows = [{"amount": 10.0}, {"amount": 2.5}]
    assert nightly_total(rows) == 12.5


def test_nightly_total_empty() -> None:
    assert nightly_total([]) == 0.0


def test_nightly_total_missing_key() -> None:
    with pytest.raises(KeyError):
        nightly_total([{"qty": 1.0}])
```

#### Complete `memory.md` after this update

#### What changed

### New Project Preference CASEs

In [33]:
# New conversation 5: Project Preferences
memory_chat_7 = PersonalProjectAssistant(memory_enabled=True)
ask_and_show_memory_diff(memory_chat_7, "Now, I'll be working on Agent memory project. CSV analytics project is complete")

### User

Now, I'll be working on Agent memory project. CSV analytics project is complete

### Assistant

Noted — CSV analytics pipeline marked complete; new project: Agent memory.

Before I write anything, I need a few specifics:

1. **Scope** — what does "agent memory" mean here: conversation history store, long-term vector recall, key-value scratchpad, or all three?
2. **Storage** — Oracle (your default), or something else (SQLite/Postgres/vector DB)?
3. **Interface** — Python library, CLI, or HTTP service?
4. **Retrieval** — exact lookup, recency window, semantic search, or hybrid?
5. **Project layout** — new repo name and directory, or existing path?

Answer these and I'll scaffold it with uv, type hints, pycodestyle, and pytest.

#### Complete `memory.md` after this update

#### What changed

In [34]:
# New conversation 5: Project Preferences
memory_chat_7 = PersonalProjectAssistant(memory_enabled=True)
ask_and_show_memory_diff(memory_chat_7, "I want to build project around agent skills. Tell me about agent skills in brief.")

### User

I want to build project around agent skills. Tell me about agent skills in brief.

### Assistant

**Agent skills** — brief:

- A **skill** is a packaged, reusable capability an LLM agent can load on demand: a folder with a `SKILL.md` (name, description, when-to-use, instructions) plus optional scripts/resources.
- **Progressive disclosure**: only the name+description sit in context; the full instructions/scripts load when the agent decides the skill is relevant. Keeps context small.
- **Composition**: skills are plain files, so agents can chain them, and you can version/test them like code.
- **vs. tools/functions**: tools are callable APIs; skills are *instructions + assets* telling the agent how to do a task (often using tools).
- **vs. MCP**: MCP exposes servers/tools; skills teach procedure and can bundle scripts that call MCP tools.

Before I scaffold anything, which do you want?

1. A **skill registry/loader** (Python: discover `skills/*/SKILL.md`, parse frontmatter, expose to an agent)
2. A **skill authoring + validation** tool (lint `SKILL.md`, pycodestyle, pytest)
3. A **demo agent** that selects and runs skills
4. Something else — describe it

Also: which agent framework (or none/raw API)?

#### Complete `memory.md` after this update

#### What changed

## 5. Pros and cons of file-based memory

Using `memory.md` is a practical first memory implementation because it is simple and transparent.

| Pros | Cons |
| --- | --- |
| Human-readable and easy to debug | Harder to search as it grows |
| No database required | Concurrent writes can conflict |
| Easy for users to edit or delete | Weak for multi-user access control |
| Predictable four-section structure | Adds prompt tokens over time |

## Takeaway

Conversation history remembers only within one chat. File-backed memory is explicit, inspectable, and optional: turn off `memory_enabled` to run the same Coding Project Assistant without reading or writing prior preferences. In a production agent, use stronger schema validation and ask for consent before retaining sensitive details.